# 01 - Standard Tasks

This notebook demonstrates the simplicity of using **RTGL** for standard tasks (focusing on *directly connected tables* or tables connected by a single shortest temporal path) in **Relational Deep Learning**.

The [**RelBench**](https://relbench.stanford.edu/) framework is used as the primary source of data and tasks, leveraging its collection of pre-defined tasks to evaluate **RTGL**'s capabilities.

## Table of Contents
1. [F1 Dataset](#f1-dataset)  
    - 1.1 [Entity Classification Tasks](#f1-clas-tasks)
        - 1.1.1 [driver-dnf](#driver-dnf)  
        - 1.1.2 [driver-top3](#driver-top3)  
    - 1.2 [Entity Regression Tasks](#f1-reg-tasks)  
        - 1.2.1 [driver-position](#driver-position)  
2. [Stack-Exchange Q&A Website Dataset](#stack-exchange-dataset)
    - 2.1 [Entity Classification Tasks](#stack-clas-tasks)  
        - 2.1.1 [user-engagement](#user-engagement)  
        - 2.1.2 [user-badge](#user-badge)  
    - 2.2 [Entity Regression Tasks](#stack-reg-tasks)  
        - 2.2.1 [post-votes](#post-votes)  
    - 2.3 [Link Prediction Tasks](#stack-link-tasks)  
        - 2.3.1 [user-post-comment](#user-post-comment)    
3. [Amazon e-commerce Dataset](#amazon-dataset)
    - 3.1 [Entity Classification Tasks](#amazon-clas-tasks)
        - 3.1.1 [user-churn](#user-churn)
        - 3.1.2 [item-churn](#item-churn)
    - 3.2 [Entity Regression Tasks](#amazon-reg-tasks)
        - 3.2.1 [user-ltv](#user-ltv)
    - 3.3 [Link Prediction Tasks](#amazon-link-tasks)
        - 3.3.1 [user-item-purchase](#user-item-purchase)
        - 3.3.2 [user-item-rate](#user-item-rate)

In [69]:
%load_ext autoreload
%autoreload 2

from experiments.utils import load_dataset_rb, load_task_rb, check_correctness

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. F1 Dataset <a id="f1-dataset"></a>

In this section, we attempt to generate the same tasks from the `F1 Dataset` which are already pre-defined in *RelBench*.

In [70]:
dataset_f1 = load_dataset_rb(name="rel-f1")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

### 1.1 Entity Classification Tasks <a id="f1-clas-tasks"></a>

#### 1.1.1 driver-dnf <a id="driver-dnf"></a>

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month.

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [71]:
task_f1_dnf = load_task_rb(dataset_f1, "driver-dnf")

In [72]:
rtgl_query = """
    PREDICT MAX(results.statusId, 0, 30, DAYS) != 1
    FOR EACH drivers.*
    WHERE COUNT(results.*, -365, 0, DAYS) != 0
    ASSUMING MAX(results.statusID, 0, 30, DAYS) IS NOT NULL 
    ;
"""

In [73]:
# TRAIN

check_correctness(dataset_f1, task_f1_dnf, rtgl_query, split="train")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.71 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp   fk label     _merge
0     2000-02-27    1     1  left_only
42    2001-02-21    3     1  left_only
50    2003-02-11    3     0  left_only
68    2001-02-21    7     1  left_only
102   2004-06-05    9     1  left_only
...          ...  ...   ...        ...
11406 1950-08-18  802     0  left_only
11407 1950-05-20  803     1  left_only
11408 1953-05-04  804     1  left_only
11409 1954-05-29  805     1  left_only
11410 1956-01-19  806     1  left_only

[1022 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
        timestamp   fk label _merge
1     2000-03-28    1     1   both
2     2000-04-27    1     1   both
3     2000-05-27    1     1   both
4     2000-06-2

In [74]:
# VAL

check_correctness(dataset_f1, task_f1_dnf, rtgl_query, split="val")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.06 seconds
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp  fk label     _merge
0   2007-02-20   0     0  left_only
34  2006-02-25   2     1  left_only
78  2007-02-20   4     1  left_only
88  2007-10-18   5     1  left_only
90  2008-03-16   6     1  left_only
117 2006-07-25   8     1  left_only
130 2008-03-16   9     1  left_only
157 2008-03-16  11     1  left_only
236 2007-02-20  15     1  left_only
298 2005-03-02  18     1  left_only
299 2007-02-20  18     1  left_only
309 2007-05-21  19     0  left_only
392 2005-04-01  23     0  left_only
411 2005-04-01  24     0  left_only
412 2007-02-20  24     1  left_only
420 2006-02-25  25     1  left_only
434 2005-03-02  26     1  left_only
455 2007-07-20  27     1  left_only
456 2006-07-25  28     1  left_only
505 2005-03-02  32   

In [75]:
# TEST

check_correctness(dataset_f1, task_f1_dnf, rtgl_query, split="test")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.07 seconds
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp   fk label     _merge
123 2012-02-20    7     0  left_only
236 2013-03-16   15     1  left_only
360 2010-06-30   28     1  left_only
364 2010-03-02   29     0  left_only
392 2010-08-29   31     1  left_only
394 2010-03-02   36     1  left_only
412 2011-03-27   38     1  left_only
463 2012-02-20  153     1  left_only
502 2010-03-02  807     1  left_only
511 2012-02-20  807     1  left_only
522 2010-03-02  808     1  left_only
550 2010-03-02  809     1  left_only
559 2010-03-02  810     1  left_only
582 2010-03-02  811     1  left_only
588 2011-03-27  812     1  left_only
608 2011-03-27  813     1  left_only
628 2011-03-27  814     1  left_only
648 2011-03-27  815     1  left_only
658 2011-06-25  816     1  left_only


To demonstrate why the mismatch appears, let's examine the original *RelBench* query for this task:
```sql
SELECT
    t.timestamp as date,
    re.driverId as driverId,
    MAX(CASE WHEN re.statusId != 1 THEN 1 ELSE 0 END) AS did_not_finish
FROM
    timestamp_df t
LEFT JOIN
    results re
ON
    re.date <= t.timestamp + INTERVAL '{self.timedelta}'
    and re.date  > t.timestamp
WHERE
    -- Data Leakage: missing upper bound(<= t.timestamp)
    re.driverId IN (
        SELECT DISTINCT driverId
        FROM results
        WHERE date > t.timestamp - INTERVAL '1 year'
    )
GROUP BY t.timestamp, re.driverId
```

It easily to note that *RelBench* team forgot to add an upper bound for the driver filtering.  

This allows `data leakage` from the future, as the query can see if a driver will participate in a race after the prediction timestamp.

#### 1.1.2 driver-top3 <a id="driver-top3"></a>

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [76]:
task_f1_top3 = load_task_rb(dataset_f1, "driver-top3")

In [77]:
rtgl_query = """
    PREDICT MIN(qualifying.position, 0, 30, DAYS) <= 3
    FOR EACH drivers.*
    ASSUMING MIN(qualifying.position, 0, 30, DAYS) IS NOT NULL;
"""

In [78]:
# TRAIN

check_correctness(dataset_f1, task_f1_top3, rtgl_query, split="train")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'qualifying' has multiple temporal paths to parent table 'drivers'. Using the shortest one: qualifying.driverid -> drivers.driverid


SQL query executed in 0.19 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk label _merge
0    2000-02-27    1     0   both
1    2000-03-28    1     0   both
2    2000-09-24    1     0   both
3    2001-09-19    1     0   both
4    2002-02-16    1     0   both
...         ...  ...   ...    ...
1348 1994-08-27  112     0   both
1349 1994-08-27  113     0   both
1350 1994-09-26  114     0   both
1351 1994-10-26  114     0   both
1352 1994-10-26  115     0   both

[1353 rows x 4 columns]
------------------- END TRAIN ---------------------


In [79]:
# VAL

check_correctness(dataset_f1, task_f1_top3, rtgl_query, split="val")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'qualifying' has multiple temporal paths to parent table 'drivers'. Using the shortest one: qualifying.driverid -> drivers.driverid


SQL query executed in 0.04 seconds
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp  fk label _merge
0   2007-02-20   0     0   both
1   2007-03-22   0     1   both
2   2007-04-21   0     0   both
3   2007-05-21   0     1   both
4   2007-06-20   0     1   both
..         ...  ..   ...    ...
583 2005-05-31  39     0   both
584 2005-06-30  39     0   both
585 2005-05-31  40     0   both
586 2005-08-29  41     0   both
587 2005-09-28  41     0   both

[588 rows x 4 columns]
------------------- END VAL ---------------------


In [80]:
# TEST

check_correctness(dataset_f1, task_f1_top3, rtgl_query, split="test")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'qualifying' has multiple temporal paths to parent table 'drivers'. Using the shortest one: qualifying.driverid -> drivers.driverid


SQL query executed in 0.06 seconds
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk label _merge
0   2010-03-02    0     0   both
1   2010-04-01    0     0   both
2   2010-05-01    0     1   both
3   2010-05-31    0     1   both
4   2010-06-30    0     0   both
..         ...  ...   ...    ...
721 2013-03-16  819     0   both
722 2013-03-16  820     0   both
723 2013-03-16  821     0   both
724 2013-03-16  822     0   both
725 2013-03-16  823     0   both

[726 rows x 4 columns]
------------------- END TEST ---------------------


### 1.2 Entity Regression Tasks <a id="f1-reg-tasks"></a>

#### 1.2.1 driver-position <a id="driver-position"></a>

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [81]:
task_f1_pos = load_task_rb(dataset_f1, "driver-position")

In [82]:
rtgl_query = """
    PREDICT AVG(results.positionOrder, 0, 60, DAYS)
    FOR EACH drivers.*;
"""

In [83]:
# TRAIN

check_correctness(dataset_f1, task_f1_pos, rtgl_query, split="train")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.24 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk    label _merge
0    2000-01-28    1  14.0000   both
1    2000-03-28    1  17.6667   both
2    2000-05-27    1  13.7500   both
3    2000-07-26    1  15.0000   both
4    2000-09-24    1  19.5000   both
...         ...  ...      ...    ...
7448 1950-06-19  801   6.0000   both
7449 1950-08-18  802   2.0000   both
7450 1953-04-04  804  17.0000   both
7451 1954-05-29  805  30.0000   both
7452 1956-01-19  806   6.0000   both

[7453 rows x 4 columns]
------------------- END TRAIN ---------------------


In [84]:
# VAL

check_correctness(dataset_f1, task_f1_pos, rtgl_query, split="val")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.04 seconds
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk    label _merge
0   2007-02-20    0   2.3333   both
1   2007-04-21    0   1.5000   both
2   2007-06-20    0   4.0000   both
3   2007-08-19    0   6.2000   both
4   2007-10-18    0   7.0000   both
..         ...  ...      ...    ...
494 2009-08-08  152  17.4000   both
495 2009-10-07  152  17.0000   both
496 2009-08-08  153  16.8000   both
497 2009-10-07  153  15.5000   both
498 2009-10-07  154   8.0000   both

[499 rows x 4 columns]
------------------- END VAL ---------------------


In [85]:
# TEST

check_correctness(dataset_f1, task_f1_pos, rtgl_query, split="test")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'results' has multiple temporal paths to parent table 'drivers'. Using the shortest one: results.driverid -> drivers.driverid


SQL query executed in 0.05 seconds
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk    label _merge
0   2010-03-02    0   4.2500   both
1   2010-05-01    0   4.6000   both
2   2010-06-30    0   8.6667   both
3   2010-08-29    0  10.0000   both
4   2010-10-28    0   3.0000   both
..         ...  ...      ...    ...
755 2016-05-29  835  17.0000   both
756 2016-01-30  836  19.0000   both
757 2016-03-30  836  19.2500   both
758 2016-05-29  836  18.0000   both
759 2016-03-30  837  10.0000   both

[760 rows x 4 columns]
------------------- END TEST ---------------------


## 2. Stack-Exchange Q&A Website Dataset <a id="stack-exchange-dataset"></a>

In this section, we attempt to generate the same tasks from the `Stack-Exchange Q&A Website Dataset` which are already pre-defined in *RelBench*.

In [86]:
dataset_stack = load_dataset_rb(name="rel-stack")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

### 2.1 Entity Classification Tasks <a id="stack-clas-tasks"></a>

#### 2.1.1 user-engagement <a id="user-engagement"></a>

Task Description: For each user predict if a user will make any votes, posts, or comments in the next 3 months. 

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [87]:
task_stack_engage = load_task_rb(dataset_stack, "user-engagement")

In [88]:
rtgl_query = """
     PREDICT COUNT(votes.*, 0, 91, DAYS) != 0
          OR COUNT(posts.*, 0, 91, DAYS) != 0
          OR COUNT(comments.*, 0, 91, DAYS) != 0
     FOR EACH users.*
     WHERE COUNT(votes.*, -inf, 0, DAYS) != 0
           OR COUNT(posts.*, -inf, 0, DAYS) != 0
           OR COUNT(comments.*, -inf, 0, DAYS) != 0;
"""

In [89]:
# TRAIN

check_correctness(dataset_stack, task_stack_engage, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 3 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple temporal paths to parent table 'users'. Using the shortest one: posts.owneruserid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'users'. Using the shortest one: votes.userid -> users.id


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 2.56 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
          timestamp      fk label     _merge
1249    2010-07-15      37     1  left_only
7790    2010-07-15     270     1  left_only
20439   2010-01-14     855     0  left_only
20440   2010-04-15     855     0  left_only
20441   2010-07-15     855     0  left_only
...            ...     ...   ...        ...
1360845 2020-07-02  251501     0  left_only
1360846 2019-10-03  252540     0  left_only
1360847 2020-01-02  252540     0  left_only
1360848 2020-04-02  252540     0  left_only
1360849 2020-07-02  252540     0  left_only

[1613 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk label _merge
0       2010-10-14       0     0   both
1       201

To demonstrate why the mismatch appears, let's examine the user with `id=37`.

In [90]:
users_table_df = dataset_stack.get_db().table_dict['users'].df
users_table_df[users_table_df['Id'] == 37]

,Id,AccountId,DisplayName,Location,WebsiteUrl,AboutMe,CreationDate
37,37,9084.0,hadley,"Houston, TX",http://hadley.nz,I'm an assistant professor of Statistics at Ri...,2010-07-19 19:13:09.870


In the resulting table, we can observe a record with `fk=37` and `timestamp=2010-07-15`. However, in the original users table, `CreationDate=2010-07-19` for this user.

This confirms a `data leakage` from the future, as the task includes users who had not yet been created at the time of the prediction.

In [91]:
# VAL

check_correctness(dataset_stack, task_stack_engage, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 3 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple temporal paths to parent table 'users'. Using the shortest one: posts.owneruserid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'users'. Using the shortest one: votes.userid -> users.id


SQL query executed in 0.33 seconds
------------------- START VAL -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp      fk label     _merge
85825 2020-10-01  247497     0  left_only
85826 2020-10-01  247505     1  left_only
85827 2020-10-01  248036     0  left_only
85828 2020-10-01  248388     0  left_only
85829 2020-10-01  249024     1  left_only
85830 2020-10-01  249067     0  left_only
85831 2020-10-01  249601     0  left_only
85832 2020-10-01  250052     1  left_only
85833 2020-10-01  250509     1  left_only
85834 2020-10-01  250586     1  left_only
85835 2020-10-01  251501     0  left_only
85836 2020-10-01  252540     0  left_only
85837 2020-10-01  252669     0  left_only
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
        timestamp      fk label _merge
0     2020-10-01    

In [92]:
# TEST

check_correctness(dataset_stack, task_stack_engage, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 3 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple temporal paths to parent table 'users'. Using the shortest one: posts.owneruserid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'users'. Using the shortest one: votes.userid -> users.id


SQL query executed in 0.34 seconds
------------------- START TEST -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
        timestamp      fk label _merge
0     2021-01-01       0     1   both
1     2021-01-01       2     0   both
2     2021-01-01       4     0   both
3     2021-01-01       5     0   both
4     2021-01-01       6     0   both
...          ...     ...   ...    ...
88132 2021-01-01  255341     0   both
88133 2021-01-01  255347     0   both
88134 2021-01-01  255351     1   both
88135 2021-01-01  255354     0   both
88136 2021-01-01  255358     1   both

[88137 rows x 4 columns]
------------------- END TEST ---------------------


#### 2.1.2 user-badge <a id="user-badge"></a>

Task Description: For each user predict if a user will receive a new badge in the next 3 months. 

In [93]:
task_stack_badge = load_task_rb(dataset_stack, "user-badge")

In [94]:
rtgl_query = """
    PREDICT COUNT(badges.*, 0, 91, DAYS) != 0
    FOR EACH users.*;
"""

In [95]:
# TRAIN

check_correctness(dataset_stack, task_stack_badge, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.71 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk label _merge
0       2010-10-14       0     0   both
1       2011-01-13       0     0   both
2       2011-04-14       0     0   both
3       2011-07-14       0     0   both
4       2011-10-13       0     0   both
...            ...     ...   ...    ...
3386271 2020-07-02  239940     0   both
3386272 2020-07-02  239941     0   both
3386273 2020-07-02  239942     0   both
3386274 2020-07-02  239943     0   both
3386275 2020-07-02  239944     1   both

[3386276 rows x 4 columns]
------------------- END 

In [96]:
# VAL

check_correctness(dataset_stack, task_stack_badge, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.15 seconds
------------------- START VAL -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2020-10-01       0     0   both
1      2020-10-01       1     0   both
2      2020-10-01       2     0   both
3      2020-10-01       3     0   both
4      2020-10-01       4     0   both
...           ...     ...   ...    ...
247393 2020-10-01  247393     0   both
247394 2020-10-01  247394     1   both
247395 2020-10-01  247395     0   both
247396 2020-10-01  247396     0   both
247397 2020-10-01  247397     1   both

[247398 rows x 4 columns]
------------------- END VAL -----------

In [97]:
# TEST

check_correctness(dataset_stack, task_stack_badge, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.14 seconds
------------------- START TEST -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2021-01-01       0     0   both
1      2021-01-01       1     0   both
2      2021-01-01       2     0   both
3      2021-01-01       3     0   both
4      2021-01-01       4     1   both
...           ...     ...   ...    ...
255355 2021-01-01  255355     1   both
255356 2021-01-01  255356     0   both
255357 2021-01-01  255357     0   both
255358 2021-01-01  255358     1   both
255359 2021-01-01  255359     1   both

[255360 rows x 4 columns]
------------------- END TEST ---------

### 2.2 Entity Regression Tasks <a id="stack-reg-tasks"></a>

#### 2.2.1 post-votes <a id="post-votes"></a>

Task Description: For each user post predict how many votes it will receive in the next 3 months 

Note: Label drift in new RelBench [version](https://github.com/stanford-star/relbench/releases/tag/v3.0.1).

In [98]:
task_stack_post_votes = load_task_rb(dataset_stack, "post-votes")

In [99]:
rtgl_query = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                       AND posts.OwnerUserId IS NOT NULL
                       AND posts.OwnerUserId != -1
    ;
"""

In [100]:
# TRAIN

check_correctness(dataset_stack, task_stack_post_votes, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'posts'. Using the shortest one: votes.postid -> posts.id


SQL query executed in 1.45 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
          timestamp      fk  label     _merge
1229873 2015-04-09   98405      0  left_only
1229874 2015-07-09   98405      0  left_only
1229875 2015-10-08   98405      0  left_only
1229876 2016-01-07   98405      0  left_only
1229877 2016-04-07   98405      0  left_only
...            ...     ...    ...        ...
2425533 2020-04-02  285930      0  left_only
2425534 2020-07-02  285930      0  left_only
2447788 2020-04-02  303318      0  left_only
2447789 2020-07-02  303318      0  left_only
2453546 2020-07-02  314184      0  left_only

[117 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk  label _merge
0       2009-04-16       0      0   both
1  

In [101]:
posts_table_df = dataset_stack.get_db().table_dict['posts'].df
posts_table_df[posts_table_df['Id'] == 98405]

,Id,OwnerUserId,PostTypeId,ParentId,OwnerDisplayName,Title,Tags,ContentLicense,Body,CreationDate
98405,98405,<NA>,1,<NA>,ckluss,Choose the correct regression model,<r><optimization><regression>,CC BY-SA 3.0,<p>The relation of two measurement values <cod...,2015-01-31 20:09:08.913


In [102]:
# VAL

check_correctness(dataset_stack, task_stack_post_votes, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'posts'. Using the shortest one: votes.postid -> posts.id


SQL query executed in 0.12 seconds
------------------- START VAL -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
         timestamp      fk  label     _merge
42646  2020-10-01   98405      0  left_only
51627  2020-10-01  116365      0  left_only
67186  2020-10-01  147709      0  left_only
86290  2020-10-01  185619      0  left_only
90495  2020-10-01  194093      0  left_only
115609 2020-10-01  244898      0  left_only
120151 2020-10-01  254145      0  left_only
124002 2020-10-01  261693      0  left_only
132398 2020-10-01  278621      0  left_only
136033 2020-10-01  285930      0  left_only
145052 2020-10-01  303318      0  left_only
150681 2020-10-01  314184      0  left_only
152470 2020-10-01  317657      0  left_only
156005 2020-10-01  324561      0  left_only
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
I

In [103]:
# TEST

check_correctness(dataset_stack, task_stack_post_votes, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 1 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'votes' has multiple temporal paths to parent table 'posts'. Using the shortest one: votes.postid -> posts.id


SQL query executed in 0.20 seconds
------------------- START TEST -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk  label _merge
0      2021-01-01       0      2   both
1      2021-01-01      19      0   both
2      2021-01-01      23      0   both
3      2021-01-01      24      0   both
4      2021-01-01      25      0   both
...           ...     ...    ...    ...
160898 2021-01-01  333883      0   both
160899 2021-01-01  333885      0   both
160900 2021-01-01  333886      0   both
160901 2021-01-01  333887      0   both
160902 2021-01-01  333891      0   both

[160903 rows x 4 columns]
------------------- END TEST ---------------------


### 2.3 Link Prediction Tasks <a id="stack-link-tasks"></a>

#### 2.3.1 user-post-comment <a id="user-post-comment"></a>

Task Description: Predict a list of existing posts that a user will comment in the next two months. 

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [104]:
task_stack_post_comm = load_task_rb(dataset_stack, "user-post-comment")

In [105]:
rtgl_query = """
    PREDICT LIST_DISTINCT(comments.PostId 
        WHERE posts.owneruserid IS NOT NULL
          AND posts.owneruserid != -1, 0, 91, DAYS)
    FOR EACH users.*;
"""

In [106]:
# TRAIN

check_correctness(dataset_stack, task_stack_post_comm, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 2 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple paths to parent table 'comments'. Using the shortest one: posts.id -> comments.postid


SQL query executed in 1.18 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp      fk                                    label     _merge
1116  2010-10-14    1141  (779, 866, 924, 1081, 1189, 2540, 2730)  left_only
1152  2010-10-14    1172             (321, 821, 1339, 1383, 2080)  left_only
1179  2010-10-14    1187                             (2289, 2851)  left_only
1297  2010-10-14    1324                              (790, 2927)  left_only
1334  2010-10-14    1365                                  (2544,)  left_only
...          ...     ...                                      ...        ...
21235 2020-07-02  246768                                (246493,)  left_only
21236 2020-07-02  246774                                (160738,)  left_only
21237 2020

To demonstrate why the mismatch appears, let's examine the user with `id=1141`.

In [107]:
users_table_df = dataset_stack.get_db().table_dict['users'].df
users_table_df[users_table_df['Id'] == 1141]

,Id,AccountId,DisplayName,Location,WebsiteUrl,AboutMe,CreationDate
1141,1141,224447.0,vqv,"Columbus, OH",http://vince.vu,NaN,2010-10-22 13:50:47.390


In the resulting table, we can observe a record with `fk=1141` and `timestamp=2010-10-14`. However, in the original users table, `CreationDate=2010-10-22` for this user.

This confirms a `data leakage` from the future, as the task includes users who had not yet been created at the time of the prediction.

In [108]:
# VAL

check_correctness(dataset_stack, task_stack_post_comm, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 2 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple paths to parent table 'comments'. Using the shortest one: posts.id -> comments.postid


SQL query executed in 0.27 seconds
------------------- START VAL -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp      fk         label     _merge
725 2020-10-01  247437     (324215,)  left_only
726 2020-10-01  247530       (3816,)  left_only
727 2020-10-01  247549     (125837,)  left_only
728 2020-10-01  247557     (219891,)  left_only
729 2020-10-01  247577     (236963,)  left_only
..         ...     ...           ...        ...
820 2020-10-01  254979     (131380,)  left_only
821 2020-10-01  255019      (92562,)  left_only
822 2020-10-01  255174  (935, 42103)  left_only
823 2020-10-01  255179     (286021,)  left_only
824 2020-10-01  255211     (310316,)  left_only

[100 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp

In [109]:
# TEST

check_correctness(dataset_stack, task_stack_post_comm, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


Validation completed with 2 semantic warning(s):
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'comments' has multiple temporal paths to parent table 'users'. Using the shortest one: comments.userid -> users.id
/mnt/personal/kolesole/RTGL/experiments/utils.py:119: [WARN] Table 'posts' has multiple paths to parent table 'comments'. Using the shortest one: posts.id -> comments.postid


SQL query executed in 0.24 seconds
------------------- START TEST -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                                     label _merge
0   2021-01-01      22                          (101761, 331974)   both
1   2021-01-01     166                                  (50110,)   both
2   2021-01-01     211                                 (333776,)   both
3   2021-01-01     221                                 (295476,)   both
4   2021-01-01     287                                   (2733,)   both
..         ...     ...                                       ...    ...
753 2021-01-01  255219                     

## 3. Amazon e-commerce Dataset <a id="amazon-dataset"></a>

In this section, we attempt to generate the same tasks from the `Amazon e-commerce Dataset` which are already pre-defined in *RelBench*.

In [110]:
dataset_amazon = load_dataset_rb(name="rel-amazon")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

### 3.1 Entity Classification Tasks <a id="amazon-clas-tasks"></a>

#### 3.1.1 user-churn <a id="user-churn"></a>

Task Description: For each user, predict 1 if the customer does not review any product in the next 3 months, and 0 otherwise.  

In [111]:
task_amazon_user_churn = load_task_rb(dataset_amazon, "user-churn")

In [112]:
rtgl_query = """
     PREDICT COUNT(review.*, 0, 91, DAYS) == 0
     FOR EACH customer.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [113]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_user_churn, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 18.03 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk label _merge
0       2008-07-10        0     1   both
1       2011-01-06        0     1   both
2       2011-07-07        0     1   both
3       2012-04-05        0     0   both
4       2012-07-05        0     0   both
...            ...      ...   ...    ...
4708378 2015-01-01  1850157     1   both
4708379 2015-01-01  1850158     1   both
4708380 2015-04-02  1850161     1   both
4708381 2014-10-02  1850171     1   both
4708382 2013-04-04  1850183     1   both

[4708383 rows x 4 columns]
------------------- END TRAIN ---------------------

In [114]:
# VAL

check_correctness(dataset_amazon, task_amazon_user_churn, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.62 seconds
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk label _merge
0      2015-10-01        3     0   both
1      2015-10-01        5     0   both
2      2015-10-01       19     1   both
3      2015-10-01       20     1   both
4      2015-10-01       21     1   both
...           ...      ...   ...    ...
409787 2015-10-01  1850080     1   both
409788 2015-10-01  1850086     1   both
409789 2015-10-01  1850102     1   both
409790 2015-10-01  1850104     1   both
409791 2015-10-01  1850149     1   both

[409792 rows x 4 columns]
----------------

In [115]:
# TEST

check_correctness(dataset_amazon, task_amazon_user_churn, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.61 seconds
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk label _merge
0      2016-01-01        2     0   both
1      2016-01-01        3     1   both
2      2016-01-01        5     0   both
3      2016-01-01       17     1   both
4      2016-01-01       23     0   both
...           ...      ...   ...    ...
351880 2016-01-01  1850119     1   both
351881 2016-01-01  1850120     1   both
351882 2016-01-01  1850121     1   both
351883 2016-01-01  1850122     1   both
351884 2016-01-01  1850134     1   both

[351885 rows x 4 columns]
---------------

#### 3.1.2 item-churn <a id="item-churn"></a>

Task Description:  For each product, predict 1 if the product does not receive any reviews in the next 3 months. 

In [116]:
task_amazon_item_churn = load_task_rb(dataset_amazon, "item-churn")

In [117]:
rtgl_query = """
     PREDICT COUNT (review.*, 0, 91, DAYS) == 0
     FOR EACH product.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [118]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_item_churn, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 8.49 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk label _merge
0       2013-01-03       0     0   both
1       2013-04-04       0     0   both
2       2013-07-04       0     0   both
3       2013-10-03       0     0   both
4       2014-01-02       0     0   both
...            ...     ...   ...    ...
2536009 2014-01-02  506009     1   both
2536010 2014-10-02  506009     0   both
2536011 2015-01-01  506009     1   both
2536012 2009-01-08  506010     1   both
2536013 2013-01-03  506010     1   both

[2536014 rows x 4 columns]
------------------- END TRAIN ---------------------


In [119]:
# VAL

check_correctness(dataset_amazon, task_amazon_item_churn, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.31 seconds
------------------- START VAL -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2015-10-01       0     0   both
1      2015-10-01       1     1   both
2      2015-10-01       2     1   both
3      2015-10-01       4     0   both
4      2015-10-01       5     0   both
...           ...     ...   ...    ...
177684 2015-10-01  505996     0   both
177685 2015-10-01  505998     0   both
177686 2015-10-01  505999     0   both
177687 2015-10-01  506000     0   both
177688 2015-10-01  506002     0   both

[177689 rows x 4 columns]
------------------- END VAL ---

In [120]:
# TEST

check_correctness(dataset_amazon, task_amazon_item_churn, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.32 seconds
------------------- START TEST -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2016-01-01       0     0   both
1      2016-01-01       3     1   both
2      2016-01-01       4     0   both
3      2016-01-01       5     1   both
4      2016-01-01       6     0   both
...           ...     ...   ...    ...
166837 2016-01-01  505996     0   both
166838 2016-01-01  505998     0   both
166839 2016-01-01  505999     0   both
166840 2016-01-01  506000     0   both
166841 2016-01-01  506002     0   both

[166842 rows x 4 columns]
------------------- END TEST -

### 3.2 Entity Regression Tasks <a id="amazon-reg-tasks"></a>

#### 3.2.1 user-ltv <a id="user-ltv"></a>

Task Description: For each user, predict the $ value of the total number of products they buy and review in the next 3 months. 

In [121]:
task_amazon_user_ltv = load_task_rb(dataset_amazon, "user-ltv")

In [122]:
rtgl_query = """
     PREDICT SUM(product.price, 0, 91, DAYS)
     FOR EACH customer.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [123]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_user_ltv, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 13.92 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk  label _merge
0       2008-07-10        0   0.00   both
1       2011-01-06        0   0.00   both
2       2011-07-07        0   0.00   both
3       2012-04-05        0   6.64   both
4       2012-07-05        0  23.99   both
...            ...      ...    ...    ...
4708378 2015-01-01  1850157   0.00   both
4708379 2015-01-01  1850158   0.00   both
4708380 2015-04-02  1850161   0.00   both
4708381 2014-10-02  1850171   0.00   both
4708382 2013-04-04  1850183   0.00   both

[4708383 rows x 4 columns]
------------------- END TRAIN ---------

In [124]:
# VAL

check_correctness(dataset_amazon, task_amazon_user_ltv, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.63 seconds
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk  label _merge
0      2015-10-01        3  27.45   both
1      2015-10-01        5  28.20   both
2      2015-10-01       19   0.00   both
3      2015-10-01       20   0.00   both
4      2015-10-01       21   0.00   both
...           ...      ...    ...    ...
409787 2015-10-01  1850080   0.00   both
409788 2015-10-01  1850086   0.00   both
409789 2015-10-01  1850102   0.00   both
409790 2015-10-01  1850104   0.00   both
409791 2015-10-01  1850149   0.00   both

[409792 rows x 4 columns]
----

In [125]:
# TEST

check_correctness(dataset_amazon, task_amazon_user_ltv, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.72 seconds
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk   label _merge
0      2016-01-01        2   20.32   both
1      2016-01-01        3    0.00   both
2      2016-01-01        5  139.64   both
3      2016-01-01       17    0.00   both
4      2016-01-01       23   49.09   both
...           ...      ...     ...    ...
351880 2016-01-01  1850119    0.00   both
351881 2016-01-01  1850120    0.00   both
351882 2016-01-01  1850121    0.00   both
351883 2016-01-01  1850122    0.00   both
351884 2016-01-01  1850134    0.00   both

[351885 rows x 4 

### 3.3 Link Prediction Tasks <a id="amazon-link-tasks"></a>

#### 3.3.1 user-item-purchase <a id="user-item-purchase"></a>

Task Description: Predict the list of distinct items each customer will purchase in the next 3 months. 

In [126]:
task_amazon_user_item_purchase = load_task_rb(dataset_amazon, "user-item-purchase")

In [127]:
rtgl_query = """
     PREDICT LIST_DISTINCT(product.*, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [128]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_user_item_purchase, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 17.62 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2010-10-07        0          (93869,)   both
2       2011-04-07        0         (297923,)   both
3       2012-01-05        0         (297644,)   both
4       2012-04-05        0         (413368,)   both
...            ...      ...               ...    ...
5112798 2014-10-02  1850157  (337213, 337946)   both
5112799 2014-10-02  1850158  (337213, 337946)   both
5112800 2015-01-01  1850161         (505350,)   b

In [129]:
# VAL

check_correctness(dataset_amazon, task_amazon_user_item_purchase, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.03 seconds
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        2                       (5251, 25178, 53881, 273172)   
1      2015-10-01        3                                   (384763, 420177)   
2      2015-10-01        5                              (26989, 70766, 81149)   
3      2015-10-01       17                                          (259178,)   
4      2015-10-01       23                                           (301

In [130]:
# TEST

check_correctness(dataset_amazon, task_amazon_user_item_purchase, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.19 seconds
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2016-01-01        0                                          (125393,)   
1      2016-01-01        2                                        (14, 48618)   
2      2016-01-01        5  (2426, 6311, 81593, 81748, 81751, 81800, 88501...   
3      2016-01-01        8                                   (228518, 410356)   
4      2016-01-01       19                                              

#### 3.3.2 user-item-rate <a id="user-item-rate"></a>

Task Description: Predict the list of distinct items each customer will purchase and give a 5 star review in the next 3 months.

In [131]:
task_amazon_user_item_rate = load_task_rb(dataset_amazon, "user-item-rate")

In [132]:
rtgl_query = """
     PREDICT LIST_DISTINCT(review.product_id WHERE review.rating == 5, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [133]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_user_item_rate, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 13.38 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2011-04-07        0         (297923,)   both
2       2012-01-05        0         (297644,)   both
3       2012-04-05        0         (413368,)   both
4       2012-10-04        0         (228080,)   both
...            ...      ...               ...    ...
3667152 2014-10-02  1850157  (337213, 337946)   both
3667153 2014-10-02  1850158  (337213, 337946)   both
3667154 2015-01-01  1850161         (505350,)   b

In [134]:
# VAL

check_correctness(dataset_amazon, task_amazon_user_item_rate, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.77 seconds
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        2                               (5251, 25178, 53881)   
1      2015-10-01        3                                   (384763, 420177)   
2      2015-10-01        5                              (26989, 70766, 81149)   
3      2015-10-01       17                                          (259178,)   
4      2015-10-01       25                                   (396864, 404

In [135]:
# TEST

check_correctness(dataset_amazon, task_amazon_user_item_rate, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.83 seconds
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                label _merge
0      2016-01-01        2                          (14, 48618)   both
1      2016-01-01        5  (2426, 81748, 81751, 81800, 204004)   both
2      2016-01-01        8                     (228518, 410356)   both
3      2016-01-01       19                                 (1,)   both
4      2016-01-01       20                            (237731,)   both
...           ...      ...                         